In [2]:
import os
import random
import math

def split_dataset_by_subject():
    # ================= 配置路径 =================
    source_dir = "/mnt/dataset4/DATASETS/fmri_pretraining/preprocess_z-score/WAR_NPYZ/ADNI_NEW/cn"
    
    output_dir = "/home/chenx/code/neurostorm_ncc/data/adni"
    train_txt_path = os.path.join(output_dir, "adni_ad_mni_train.txt")
    test_txt_path = os.path.join(output_dir, "adni_ad_mni_test.txt")
    val_txt_path = os.path.join(output_dir, "adni_ad_mni_val.txt")
    
    # 确保输出目录存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # ================= 1. 获取并整理文件 =================
    # 获取所有 .npz 文件
    all_files = [f for f in os.listdir(source_dir) if f.endswith('.npz')]
    
    # 使用字典按被试ID分组: { '003S6264': [path1, path2...], ... }
    subject_files_map = {}
    
    print("正在整理文件并按被试分组...")
    for filename in all_files:
        # 解析文件名获取被试ID
        # 示例: sub-003S6264_ses-01_task-rest... -> ID: 003S6264
        try:
            # 逻辑：找到 'sub-' 后面直到第一个 '_' 之间的字符串
            subject_id = filename.split('sub-')[1].split('_')[0]
            
            full_path = os.path.join(source_dir, filename)
            
            if subject_id not in subject_files_map:
                subject_files_map[subject_id] = []
            subject_files_map[subject_id].append(full_path)
            
        except IndexError:
            print(f"警告: 文件名格式不符合预期，已跳过: {filename}")
            continue

    # 获取所有唯一的被试ID
    unique_subjects = list(subject_files_map.keys())
    total_subjects = len(unique_subjects)
    
    print(f"总共找到 {len(all_files)} 个文件，属于 {total_subjects} 个独立被试。")

    # ================= 2. 划分数据集 (按被试) =================
    # 设置随机种子以保证结果可复现
    random.seed(42)
    random.shuffle(unique_subjects)
    
    # 计算数量 (70% : 20% : 10%)
    n_train = int(total_subjects * 0.70)
    n_test = int(total_subjects * 0.20)
    # 剩余的给验证集 (确保总和正确，避免因取整导致漏掉被试)
    n_val = total_subjects - n_train - n_test
    
    # 分割被试列表
    train_subjects = unique_subjects[:n_train]
    test_subjects = unique_subjects[n_train : n_train + n_test]
    val_subjects = unique_subjects[n_train + n_test:]
    
    print(f"划分情况 (被试数): Train={len(train_subjects)}, Test={len(test_subjects)}, Val={len(val_subjects)}")

    # ================= 3. 获取对应的文件路径列表 =================
    def get_filepaths_from_subjects(subject_list, mapping):
        paths = []
        for subj in subject_list:
            # 将该被试下的所有文件都加入列表
            paths.extend(mapping[subj])
        return paths

    train_files = get_filepaths_from_subjects(train_subjects, subject_files_map)
    test_files = get_filepaths_from_subjects(test_subjects, subject_files_map)
    val_files = get_filepaths_from_subjects(val_subjects, subject_files_map)

    # ================= 4. 写入txt文件 =================
    def write_to_txt(filepath, file_list):
        with open(filepath, 'w') as f:
            for line in file_list:
                f.write(f"{line}\n")
        print(f"已写入: {filepath} (包含 {len(file_list)} 个文件)")

    write_to_txt(train_txt_path, train_files)
    write_to_txt(test_txt_path, test_files)
    write_to_txt(val_txt_path, val_files)

    print("完成！")

if __name__ == "__main__":
    split_dataset_by_subject()

正在整理文件并按被试分组...
总共找到 991 个文件，属于 236 个独立被试。
划分情况 (被试数): Train=165, Test=47, Val=24
已写入: /home/chenx/code/neurostorm_ncc/data/adni/adni_ad_mni_train.txt (包含 692 个文件)
已写入: /home/chenx/code/neurostorm_ncc/data/adni/adni_ad_mni_test.txt (包含 200 个文件)
已写入: /home/chenx/code/neurostorm_ncc/data/adni/adni_ad_mni_val.txt (包含 99 个文件)
完成！
